In [1]:
# ============================
# Cell 1 — Core Environment
# ============================

import numpy as np

# Base random engine
rng = np.random.default_rng()

# Safety clamps
def clamp(x, lo=-1e9, hi=1e9):
    return np.minimum(np.maximum(x, lo), hi)

# Core AC‑CR operators (minimal skeleton)
def op_converge(x, rate):
    return x - rate * np.tanh(x)

def op_diverge(x, rate):
    return x + rate * np.tanh(x)

def op_noise(x, scale):
    return x + rng.normal(0, scale, size=x.shape)

# Framework wrapper
def accr_step(x, c_rate, d_rate, n_scale):
    x = op_converge(x, c_rate)
    x = op_diverge(x, d_rate)
    x = op_noise(x, n_scale)
    return clamp(x)


In [2]:
# ============================
# Test S — Structural Stability
# ============================

def run_test_S(
    size=50000,
    steps=250,
    c_rate=0.35,
    d_rate=0.40,
    n_scale=0.55
):
    x = rng.normal(0, 1, size=size)
    history = []

    for _ in range(steps):
        x = accr_step(x, c_rate, d_rate, n_scale)
        history.append(np.std(x))

    history = np.array(history)

    print("=== Test S Summary ===")
    print(f"Initial STD: {history[0]:.6f}")
    print(f"Final STD:   {history[-1]:.6f}")
    print(f"Max Deviation: {np.max(np.abs(history - history[0])):.6f}")
    print(f"Min STD: {np.min(history):.6f}")
    print(f"Max STD: {np.max(history):.6f}")

    return history

# Run Test S
history_S = run_test_S()


=== Test S Summary ===
Initial STD: 1.146561
Final STD:   16.655429
Max Deviation: 15.508868
Min STD: 1.146561
Max STD: 16.655429


In [3]:
# ==========================================
# Test S‑2 — Stability Corridor Sweep
# ==========================================

def run_test_S2(
    size=40000,
    steps=180,
    c_rates=np.linspace(0.20, 0.60, 9),
    d_rates=np.linspace(0.10, 0.50, 9),
    n_scales=np.linspace(0.10, 0.60, 9)
):
    results = []

    for c in c_rates:
        for d in d_rates:
            for n in n_scales:
                x = rng.normal(0, 1, size=size)
                for _ in range(steps):
                    x = accr_step(x, c, d, n)
                final_std = np.std(x)
                results.append((c, d, n, final_std))

    # Sort by final STD (lowest = most stable)
    results.sort(key=lambda r: r[3])

    print("=== Test S‑2: Top 12 Most Stable Regimes ===")
    for i in range(12):
        c, d, n, fs = results[i]
        print(f"{i+1:02d}: c={c:.3f}, d={d:.3f}, n={n:.3f} → Final STD={fs:.6f}")

    return results

# Run S‑2
results_S2 = run_test_S2()


=== Test S‑2: Top 12 Most Stable Regimes ===
01: c=0.600, d=0.100, n=0.100 → Final STD=0.111665
02: c=0.600, d=0.150, n=0.100 → Final STD=0.113154
03: c=0.600, d=0.200, n=0.100 → Final STD=0.115110
04: c=0.550, d=0.100, n=0.100 → Final STD=0.115583
05: c=0.600, d=0.250, n=0.100 → Final STD=0.115985
06: c=0.600, d=0.300, n=0.100 → Final STD=0.117232
07: c=0.550, d=0.150, n=0.100 → Final STD=0.117971
08: c=0.550, d=0.200, n=0.100 → Final STD=0.119348
09: c=0.600, d=0.350, n=0.100 → Final STD=0.120194
10: c=0.500, d=0.100, n=0.100 → Final STD=0.120363
11: c=0.600, d=0.400, n=0.100 → Final STD=0.121759
12: c=0.550, d=0.250, n=0.100 → Final STD=0.122102


In [4]:
# ==================================
# Test S‑3 — Noise‑Ramp Stability
# ==================================

def run_test_S3(
    size=40000,
    steps=200,
    c_rate=0.60,
    d_rate=0.10,
    n_scales=np.linspace(0.05, 0.60, 12)
):
    print("=== Test S‑3: Noise‑Ramp Stability ===")
    results = []

    for n in n_scales:
        x = rng.normal(0, 1, size=size)
        for _ in range(steps):
            x = accr_step(x, c_rate, d_rate, n)
        final_std = np.std(x)
        results.append((n, final_std))
        print(f"n={n:.3f} → Final STD={final_std:.6f}")

    return results

# Run S‑3
results_S3 = run_test_S3()


=== Test S‑3: Noise‑Ramp Stability ===
n=0.050 → Final STD=0.055737
n=0.100 → Final STD=0.111655
n=0.150 → Final STD=0.168654
n=0.200 → Final STD=0.226414
n=0.250 → Final STD=0.285812
n=0.300 → Final STD=0.347400
n=0.350 → Final STD=0.412430
n=0.400 → Final STD=0.481093
n=0.450 → Final STD=0.550965
n=0.500 → Final STD=0.624999
n=0.550 → Final STD=0.699013
n=0.600 → Final STD=0.785349


In [5]:
# ==================================
# Test S‑4 — Divergence‑Ramp Test
# ==================================

def run_test_S4(
    size=40000,
    steps=200,
    c_rate=0.60,
    n_scale=0.10,
    d_rates=np.linspace(0.05, 0.70, 14)
):
    print("=== Test S‑4: Divergence‑Ramp Stability ===")
    results = []

    for d in d_rates:
        x = rng.normal(0, 1, size=size)
        for _ in range(steps):
            x = accr_step(x, c_rate, d, n_scale)
        final_std = np.std(x)
        results.append((d, final_std))
        print(f"d={d:.3f} → Final STD={final_std:.6f}")

    return results

# Run S‑4
results_S4 = run_test_S4()


=== Test S‑4: Divergence‑Ramp Stability ===
d=0.050 → Final STD=0.110445
d=0.100 → Final STD=0.111974
d=0.150 → Final STD=0.113369
d=0.200 → Final STD=0.114402
d=0.250 → Final STD=0.116146
d=0.300 → Final STD=0.117831
d=0.350 → Final STD=0.119966
d=0.400 → Final STD=0.121927
d=0.450 → Final STD=0.124320
d=0.500 → Final STD=0.127067
d=0.550 → Final STD=0.129361
d=0.600 → Final STD=0.214984
d=0.650 → Final STD=2.576964
d=0.700 → Final STD=6.906370


In [6]:
# ==================================
# Test S‑5 — Neutral‑Line Scan
# ==================================

def run_test_S5(
    size=40000,
    steps=200,
    n_scale=0.10,
    c_rates=np.linspace(0.30, 0.70, 9),
    d_rates=np.linspace(0.05, 0.80, 16)
):
    print("=== Test S‑5: Neutral‑Line Scan ===")
    boundary = []

    for c in c_rates:
        neutral_d = None
        last_std = None

        for d in d_rates:
            x = rng.normal(0, 1, size=size)
            for _ in range(steps):
                x = accr_step(x, c, d, n_scale)
            final_std = np.std(x)

            # Detect first upward break in STD
            if last_std is not None and final_std > last_std * 1.20:
                neutral_d = d
                break

            last_std = final_std

        boundary.append((c, neutral_d))
        print(f"c={c:.3f} → neutral d ≈ {neutral_d}")

    return boundary

# Run S‑5
neutral_line = run_test_S5()


=== Test S‑5: Neutral‑Line Scan ===
c=0.300 → neutral d ≈ 0.3
c=0.350 → neutral d ≈ 0.35000000000000003
c=0.400 → neutral d ≈ 0.4
c=0.450 → neutral d ≈ 0.45
c=0.500 → neutral d ≈ 0.5
c=0.550 → neutral d ≈ 0.55
c=0.600 → neutral d ≈ 0.6000000000000001
c=0.650 → neutral d ≈ 0.6500000000000001
c=0.700 → neutral d ≈ 0.7000000000000001


In [7]:
# ================================
# Test S‑6 — Lyapunov Exponent Map
# ================================

def run_test_S6(
    size=40000,
    steps=200,
    n_scale=0.10,
    c_rates=np.linspace(0.30, 0.70, 9),
    d_rates=np.linspace(0.30, 0.70, 9)
):
    print("=== Test S‑6: Lyapunov Exponent Map ===")
    results = []

    for c in c_rates:
        for d in d_rates:
            x = rng.normal(0, 1, size=size)
            initial_std = np.std(x)

            for _ in range(steps):
                x = accr_step(x, c, d, n_scale)

            final_std = np.std(x)
            # Lyapunov-style exponent
            lam = np.log(final_std / initial_std) / steps
            results.append((c, d, lam))

    # Sort by exponent
    results.sort(key=lambda r: r[2])

    print("Top 10 most contractive (lowest λ):")
    for i in range(10):
        c, d, lam = results[i]
        print(f"{i+1:02d}: c={c:.3f}, d={d:.3f} → λ={lam:.6f}")

    print("\nTop 10 most divergent (highest λ):")
    for i in range(1, 11):
        c, d, lam = results[-i]
        print(f"{i:02d}: c={c:.3f}, d={d:.3f} → λ={lam:.6f}")

    return results

# Run S‑6
results_S6 = run_test_S6()


=== Test S‑6: Lyapunov Exponent Map ===
Top 10 most contractive (lowest λ):
01: c=0.700, d=0.350 → λ=-0.011061
02: c=0.700, d=0.300 → λ=-0.011011
03: c=0.700, d=0.400 → λ=-0.011003
04: c=0.650, d=0.300 → λ=-0.010923
05: c=0.700, d=0.450 → λ=-0.010901
06: c=0.700, d=0.550 → λ=-0.010893
07: c=0.700, d=0.500 → λ=-0.010889
08: c=0.650, d=0.350 → λ=-0.010849
09: c=0.700, d=0.600 → λ=-0.010822
10: c=0.700, d=0.650 → λ=-0.010820

Top 10 most divergent (highest λ):
01: c=0.300, d=0.700 → λ=0.021831
02: c=0.300, d=0.650 → λ=0.021184
03: c=0.350, d=0.700 → λ=0.021145
04: c=0.300, d=0.600 → λ=0.020388
05: c=0.350, d=0.650 → λ=0.020357
06: c=0.400, d=0.700 → λ=0.020231
07: c=0.300, d=0.550 → λ=0.019471
08: c=0.350, d=0.600 → λ=0.019416
09: c=0.400, d=0.650 → λ=0.019169
10: c=0.450, d=0.700 → λ=0.018629


In [8]:
# === Test S‑7: Basin Robustness Under Perturbation ===
# Evaluates λ_eff(c,d,δ) for selected contractive points under injected noise.

import numpy as np

# --- AC‑CR update function (same as earlier tests) ---
def ac_cr_step(x, c, d):
    return c * x * (1 - x) + d * (np.sin(np.pi * x)**2)

# --- Lyapunov exponent with perturbation ---
def lyapunov_with_noise(c, d, delta, x0=0.5, steps=5000, discard=1000):
    x = x0
    lyap_sum = 0.0

    for n in range(steps):
        # derivative of F wrt x
        dF = c * (1 - 2*x) + d * (np.pi * np.sin(2*np.pi*x))

        # accumulate Lyapunov after transient
        if n >= discard:
            lyap_sum += np.log(abs(dF) + 1e-12)

        # update with noise
        x = ac_cr_step(x, c, d) + np.random.uniform(-delta, delta)

        # keep x in [0,1]
        x = min(max(x, 0.0), 1.0)

    return lyap_sum / (steps - discard)

# --- Points to test (from S‑6 contractive band) ---
test_points = [
    (0.700, 0.350),
    (0.700, 0.500),
    (0.650, 0.350),
]

# --- Noise levels ---
deltas = [0.0, 1e-3, 3e-3, 1e-2]

results = []

for (c, d) in test_points:
    for delta in deltas:
        lam = lyapunov_with_noise(c, d, delta)
        results.append((c, d, delta, lam))
        print(f"(c={c:.3f}, d={d:.3f}, δ={delta:.0e}) → λ_eff={lam:.6f}")

# Store results for later analysis
results_S7 = results


(c=0.700, d=0.350, δ=0e+00) → λ_eff=-1.664941
(c=0.700, d=0.350, δ=1e-03) → λ_eff=-1.665686
(c=0.700, d=0.350, δ=3e-03) → λ_eff=-1.669195
(c=0.700, d=0.350, δ=1e-02) → λ_eff=-1.701926
(c=0.700, d=0.500, δ=0e+00) → λ_eff=-0.472942
(c=0.700, d=0.500, δ=1e-03) → λ_eff=-0.473518
(c=0.700, d=0.500, δ=3e-03) → λ_eff=-0.480217
(c=0.700, d=0.500, δ=1e-02) → λ_eff=-0.524362
(c=0.650, d=0.350, δ=0e+00) → λ_eff=-2.325352
(c=0.650, d=0.350, δ=1e-03) → λ_eff=-2.326815
(c=0.650, d=0.350, δ=3e-03) → λ_eff=-2.337983
(c=0.650, d=0.350, δ=1e-02) → λ_eff=-2.493598


In [9]:
# === Test S‑8: Basin-Boundary Deformation Under Noise ===
# Tests points near the contractive–divergent boundary to see how noise shifts λ_eff.

import numpy as np

# --- AC‑CR update function (same as earlier tests) ---
def ac_cr_step(x, c, d):
    return c * x * (1 - x) + d * (np.sin(np.pi * x)**2)

# --- Lyapunov exponent with perturbation ---
def lyapunov_with_noise(c, d, delta, x0=0.5, steps=5000, discard=1000):
    x = x0
    lyap_sum = 0.0

    for n in range(steps):
        dF = c * (1 - 2*x) + d * (np.pi * np.sin(2*np.pi*x))

        if n >= discard:
            lyap_sum += np.log(abs(dF) + 1e-12)

        x = ac_cr_step(x, c, d) + np.random.uniform(-delta, delta)
        x = min(max(x, 0.0), 1.0)

    return lyap_sum / (steps - discard)

# --- Boundary points (just inside and just outside the S‑6 ridge) ---
boundary_points = [
    (0.500, 0.550),  # slightly contractive side
    (0.450, 0.600),  # near-neutral
    (0.400, 0.650),  # slightly divergent side
    (0.350, 0.700),  # strongly divergent ridge
]

# --- Noise levels ---
deltas = [0.0, 1e-3, 3e-3, 1e-2]

results_S8 = []

print("=== Test S‑8: Basin-Boundary Deformation ===")
for (c, d) in boundary_points:
    for delta in deltas:
        lam = lyapunov_with_noise(c, d, delta)
        results_S8.append((c, d, delta, lam))
        print(f"(c={c:.3f}, d={d:.3f}, δ={delta:.0e}) → λ_eff={lam:.6f}")


=== Test S‑8: Basin-Boundary Deformation ===
(c=0.500, d=0.550, δ=0e+00) → λ_eff=-0.764451
(c=0.500, d=0.550, δ=1e-03) → λ_eff=-0.767187
(c=0.500, d=0.550, δ=3e-03) → λ_eff=-0.795012
(c=0.500, d=0.550, δ=1e-02) → λ_eff=-0.802590
(c=0.450, d=0.600, δ=0e+00) → λ_eff=-0.172426
(c=0.450, d=0.600, δ=1e-03) → λ_eff=-0.173665
(c=0.450, d=0.600, δ=3e-03) → λ_eff=-0.185494
(c=0.450, d=0.600, δ=1e-02) → λ_eff=-0.324925
(c=0.400, d=0.650, δ=0e+00) → λ_eff=0.068487
(c=0.400, d=0.650, δ=1e-03) → λ_eff=0.064659
(c=0.400, d=0.650, δ=3e-03) → λ_eff=0.083381
(c=0.400, d=0.650, δ=1e-02) → λ_eff=0.092192
(c=0.350, d=0.700, δ=0e+00) → λ_eff=0.367888
(c=0.350, d=0.700, δ=1e-03) → λ_eff=0.347091
(c=0.350, d=0.700, δ=3e-03) → λ_eff=0.363569
(c=0.350, d=0.700, δ=1e-02) → λ_eff=0.366476


In [10]:
# === Test S‑9: Global Hitting & Return Statistics ===
# Tracks how often noisy trajectories starting near the divergent ridge
# enter the contractive basin and how long they remain.

import numpy as np

# --- AC‑CR update function ---
def ac_cr_step(x, c, d):
    return c * x * (1 - x) + d * (np.sin(np.pi * x)**2)

# --- Contractive basin definition (from S‑6/S‑7) ---
# A point is considered "in the basin" if λ_eff < 0 for its (c,d).
# For S‑9 we approximate this by checking if x stays in a contractive region of state-space.
# We use a simple threshold: |dF| < 1  (local contraction)
def in_basin(x, c, d):
    dF = c * (1 - 2*x) + d * (np.pi * np.sin(2*np.pi*x))
    return abs(dF) < 1.0

# --- Global hitting test ---
def hitting_return_stats(c, d, delta, steps=20000):
    x = np.random.uniform(0, 1)  # random initial state
    hits = 0
    consecutive = 0
    max_consecutive = 0

    for n in range(steps):
        # update with noise
        x = ac_cr_step(x, c, d) + np.random.uniform(-delta, delta)
        x = min(max(x, 0.0), 1.0)

        inside = in_basin(x, c, d)

        if inside:
            hits += 1
            consecutive += 1
            max_consecutive = max(max_consecutive, consecutive)
        else:
            consecutive = 0

    hit_rate = hits / steps
    return hit_rate, max_consecutive

# --- Starting points near divergent ridge ---
ridge_points = [
    (0.400, 0.650),
    (0.350, 0.700),
    (0.300, 0.700),
]

# --- Noise levels ---
deltas = [1e-3, 3e-3, 1e-2]

print("=== Test S‑9: Global Hitting & Return Statistics ===")
results_S9 = []

for (c, d) in ridge_points:
    for delta in deltas:
        hit_rate, max_run = hitting_return_stats(c, d, delta)
        results_S9.append((c, d, delta, hit_rate, max_run))
        print(f"(c={c:.3f}, d={d:.3f}, δ={delta:.0e}) → hit_rate={hit_rate:.4f}, max_run={max_run}")


=== Test S‑9: Global Hitting & Return Statistics ===
(c=0.400, d=0.650, δ=1e-03) → hit_rate=0.2890, max_run=1
(c=0.400, d=0.650, δ=3e-03) → hit_rate=0.3154, max_run=1
(c=0.400, d=0.650, δ=1e-02) → hit_rate=0.3207, max_run=1
(c=0.350, d=0.700, δ=1e-03) → hit_rate=0.1737, max_run=1
(c=0.350, d=0.700, δ=3e-03) → hit_rate=1.0000, max_run=20000
(c=0.350, d=0.700, δ=1e-02) → hit_rate=0.9999, max_run=19997
(c=0.300, d=0.700, δ=1e-03) → hit_rate=0.1757, max_run=1
(c=0.300, d=0.700, δ=3e-03) → hit_rate=0.1769, max_run=1
(c=0.300, d=0.700, δ=1e-02) → hit_rate=0.1869, max_run=1
